In [ ]:
pip install pandas openpyxl

In [ ]:
import os
import time
from datetime import datetime
import pandas as pd
from openai import OpenAI

MODEL = "openai/gpt-5.2"     # OpenRouter model identifier     
MAX_OUTPUT_TOKENS = 10000
TEMPERATURE = 0.1
TOP_P = 1

INPUT_EXCEL = "Problems_for_ experiment.xlsx"
OUTPUT_CSV = "generated_diagrams.csv"
PUML_FOLDER = "puml_outputs"
SCENARIO_ID_COL = "ScenarioID"
SCENARIO_TEXT_COL = "ScenarioDescription"

os.makedirs(PUML_FOLDER, exist_ok=True)

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

if not os.getenv("OPENROUTER_API_KEY"):
    raise ValueError("OPENROUTER_API_KEY environment variable not set.")

In [ ]:
def trim_to_enduml(text: str) -> str:
    if "@enduml" in text:
        return text.split("@enduml")[0] + "@enduml"
    return text


def build_prompt(requirements: str) -> str:
    return f"""
You are an expert software engineer specialized in UML modeling and requirements analysis.

###Task:
Generate a UML Activity Diagram in valid PlantUML format from the given requirements.

Let's think step-by-step from the requirements to:

Step 1: Identify Actors
- Extract all roles (e.g., User, Admin, Inventory User, System, IT Staff) explicitly or implicitly mentioned in the requirements.

Step 2: Identify Main Activities
- Extract all key operations/actions in sequential order as performed by each actor. (e.g., transfer asset, approve request, edit asset, authenticate user)

Step 3: Identify Decisions and Conditions
- Detect all conditions such as:
  - approvals
  - permissions
  - alternative flows
  - validation checks
  - error conditions

Step 4: Identify Relationships
- Determine dependencies between processes
- Identify sequences and possible parallel flows

Step 5: Construct Control Flow
- Organize activities into a logical sequence
- Insert decision nodes where required
- Add loops if implied

Step 6: Assign Swimlanes
- Map each activity to the correct actor
- For Single actor generate flat activity diagram
- For Multiple actors generate swimlanes in activity diagram

Step 7: PlantUML Generation:
Using the above analysis, generate the PlantUML code following these rules:
- Use swimlanes (|Actor|) for multi-user scenarios
- Use `if (...) then (yes/no)` for decisions
- Use `fork` / `fork again` / `end fork` for parallel flows
- Use `repeat` / `repeat while` or `while` for loops
- First swimlane must be added after @startuml and before 'start'.
  
Step 8: Optimize Diagram
- Merge redundant steps
- Keep diagram readable
- Ensure logical consistency
- Every activity in the diagram must be traceable to the requirement.

###Requirements:
{requirements}

###Output:
One PlantUML code block only (from '@startuml' to '@enduml')
"""

In [ ]:
def generate_activity_diagram(prompt: str):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        max_tokens=MAX_OUTPUT_TOKENS,
        temperature=TEMPERATURE,
        top_p=TOP_P
    )

    text = response.choices[0].message.content or ""
    text = trim_to_enduml(text)

    usage = getattr(response, "usage", None)
    usage_info = {
        "input_tokens": getattr(usage, "prompt_tokens", None),
        "output_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
        "timestamp": datetime.now().isoformat(),
        "model": MODEL
    }

    return text, usage_info

In [ ]:
# =========================
# Load Excel
# =========================
df = pd.read_excel(INPUT_EXCEL)

results = []

print(f"Total scenarios found: {len(df)}")

for idx, row in df.iterrows():
    scenario_id = str(row[SCENARIO_ID_COL])
    scenario_text = str(row[SCENARIO_TEXT_COL])

    print(f"\nProcessing Scenario: {scenario_id}")

    try:
        prompt = build_prompt(scenario_text)
        output_text, usage = generate_activity_diagram(prompt)

        # =========================
        # Save .puml file
        # =========================
        puml_filename = f"{scenario_id}.puml"
        puml_path = os.path.join(PUML_FOLDER, puml_filename)

        with open(puml_path, "w", encoding="utf-8") as f:
            f.write(output_text)

        # =========================
        # Store results for CSV
        # =========================
        results.append({
            "ScenarioID": scenario_id,
            "ScenarioDescription": scenario_text,
            "GeneratedPlantUML": output_text,
            "InputTokens": usage["input_tokens"],
            "OutputTokens": usage["output_tokens"],
            "TotalTokens": usage["total_tokens"],
            "Model": usage["model"],
            "Timestamp": usage["timestamp"],
            "PUML_File": puml_path
        })

        time.sleep(0.5)

    except Exception as e:
        print(f" Error in Scenario {scenario_id}: {e}")

        results.append({
            "ScenarioID": scenario_id,
            "ScenarioDescription": scenario_text,
            "GeneratedPlantUML": "ERROR",
            "InputTokens": None,
            "OutputTokens": None,
            "TotalTokens": None,
            "Model": MODEL,
            "Timestamp": datetime.now().isoformat(),
            "PUML_File": None
        })

# =========================
# Save CSV
# =========================
results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_CSV, index=False)

print("\n Processing complete.")
print(f"CSV saved to: {OUTPUT_CSV}")
print(f"PUML files folder: {PUML_FOLDER}")